# NISAR, Каховка v4: два незалежні спостерігачі

**Проблема.** На ділянці кожен знімок NISAR забитий радіоперешкодами, які змінюються від дати до дати. Будь-яке порівняння двох знімків одного треку змішує зміну поверхні зі зміною перешкод, і відокремити одне від одного всередині одного треку неможливо (розділ 7: спроба «відняти перешкоду моделлю» провалилась).

**Ідея розв'язку.** Над ділянкою літають **два незалежні треки**: 143_A о 02:39 UTC (висхідний) і 108_D о 16:46 UTC (низхідний), з різницею в 3 доби. Поверхня в них одна, а перешкоди, геометрія й спекл різні. Тож **те, у чому вони збігаються, — це поверхня; те, у чому розходяться, — шум.** Це той самий принцип, яким кріоелектронна мікроскопія визначає роздільність карти: дві незалежні половини даних порівнюються **за кожною просторовою частотою** (Fourier Ring/Shell Correlation; van Heel 1987, Rosenthal & Henderson 2003).

**Що рахуємо**
1. **FRC(k)** сезонних змін двох треків: на яких масштабах сигнал справжній. Плюс критерії 3σ і нуль-модель зсуву.
2. **Якість кожної дати:** FRC майже одночасних знімків двох треків.
3. **Оптимальне злиття** двох треків фільтром $w(k)=\sqrt{2\,\mathrm{FRC}/(1+\mathrm{FRC})}$ і карта «шуму» (половина різниці).
4. **Локальна карта узгодженості:** де саме на ділянці радар бачить правду.
5. **Ритм:** кореляція часових рядів двох треків у кожному пікселі з перестановочним тестом і FDR, плюс часові моди SVD спільного сигналу — типи сезонного ритму полів.

In [ ]:
from pathlib import Path
from datetime import datetime
import json, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from scipy.ndimage import median_filter, binary_erosion, zoom
from pyproj import Transformer
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_origin
from IPython.display import display

warnings.filterwarnings('ignore', category=RuntimeWarning)
plt.rcParams.update({'figure.dpi': 100, 'axes.titlesize': 10})
DATA_DIR = Path('..') / 'data'
if not (DATA_DIR / 'h5').exists():
    DATA_DIR = next(p / 'data' for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'h5').exists())
CACHE = DATA_DIR / 'cache_ts'; OUT = DATA_DIR / 'analysis_v4'; OUT.mkdir(exist_ok=True)
TRACKS = ['143_A_025', '108_D_065']; A_, B_ = TRACKS
BBOX, EPSG, RES = (33.4, 46.9, 35.2, 47.8), 32636, 80.0
PX_KM = RES / 1000
rng = np.random.default_rng(0)

_t = Transformer.from_crs(4326, EPSG, always_xy=True)
_X, _Y = _t.transform([BBOX[0], BBOX[0], BBOX[2], BBOX[2]], [BBOX[1], BBOX[3], BBOX[1], BBOX[3]])
X = np.arange(np.floor((min(_X) - 40) / 80) * 80 + 40, max(_X), RES)
Y = np.arange(np.ceil((max(_Y) - 40) / 80) * 80 + 40, min(_Y), -RES)
H, W = len(Y), len(X)
TRANSFORM = from_origin(X[0] - 40, Y[0] + 40, RES, RES)
EXTENT = [(X[0] - 40) / 1e3, (X[-1] + 40) / 1e3, (Y[-1] - 40) / 1e3, (Y[0] + 40) / 1e3]

S = {}
for t in TRACKS:
    files = sorted(CACHE.glob(f'{t}_*.npz'))
    assert files, 'Немає кешу: спершу запустіть nisar_kakhovka_timeseries.ipynb'
    d = {'time': [], 'hh': [], 'hv': [], 'ver': []}
    for f in files:
        z = np.load(f); d['time'].append(datetime.strptime(f.stem.split('_')[-1], '%Y%m%dT%H%M'))
        d['ver'].append(json.loads(str(z['meta']))['version']); d['hh'].append(z['hhB'].astype('f4')); d['hv'].append(z['hvB'].astype('f4'))
    S[t] = {k: (np.stack(v) if k in ('hh', 'hv') else v) for k, v in d.items()}
    print(t, len(files), 'дат')

def to_grid(p, rs, nodata):
    dst = np.full((H, W), nodata, 'u1')
    with rasterio.open(p) as s:
        reproject(rasterio.band(s, 1), dst, dst_transform=TRANSFORM, dst_crs=f'EPSG:{EPSG}', src_nodata=s.nodata if s.nodata is not None else nodata, dst_nodata=nodata, resampling=rs)
    return dst
WC = to_grid(DATA_DIR / 'aux' / 'worldcover_2021_aoi.tif', Resampling.mode, 0)
J = to_grid(DATA_DIR / 'aux' / 'jrc_occurrence_aoi.tif', Resampling.average, 255).astype('f4'); J[J > 100] = np.nan; J0 = np.nan_to_num(J, nan=0)
LAND = (J0 < 5) & (WC != 80); CROP = (WC == 40) & LAND; BED = J0 >= 50
BOTH = np.isfinite(S[A_]['hh']).all(0) & np.isfinite(S[B_]['hh']).all(0)
print(f'Ділянка, яку бачать обидва треки на всіх датах: {BOTH.sum() * PX_KM**2:,.0f} км²')

# Пари майже одночасних знімків (≤ 5 діб)
PAIRS = []
for i, ta in enumerate(S[A_]['time']):
    j = int(np.argmin([abs((tb - ta).total_seconds()) for tb in S[B_]['time']]))
    if abs((S[B_]['time'][j] - ta).days) <= 5: PAIRS.append((i, j))
print('Пари дат:', ', '.join(f"{S[A_]['time'][i]:%d.%m}/{S[B_]['time'][j]:%d.%m}" for i, j in PAIRS))

## 1. Проблема наочно: той самий тиждень, два треки

Зліва і в центрі — HH двох треків з різницею в 3 доби. Справа — їхня різниця. Поля, русло й дно однакові, але поверх них лежать різні розмиті плями перешкод. Саме ці плями й заважають бачити зміни в межах одного треку.

In [ ]:
i, j = PAIRS[0]
fig, axs = plt.subplots(1, 3, figsize=(18, 5.2), constrained_layout=True)
for ax, a, ttl in ((axs[0], S[A_]['hh'][i], f'{A_} {S[A_]["time"][i]:%d.%m %H:%M} UTC'), (axs[1], S[B_]['hh'][j], f'{B_} {S[B_]["time"][j]:%d.%m %H:%M} UTC')):
    ax.imshow(a, cmap='gray', vmin=-20, vmax=-2, extent=EXTENT); ax.set_title(f'HH {ttl}')
q = axs[2].imshow(median_filter(np.nan_to_num(S[A_]['hh'][i] - S[B_]['hh'][j]), 5), cmap='coolwarm', vmin=-6, vmax=6, extent=EXTENT)
axs[2].set_title('різниця треків: розмиті плями — перешкоди, а не поверхня'); fig.colorbar(q, ax=axs[2], shrink=.8, label='дБ')
fig.savefig(OUT / 'problem_two_tracks.png', dpi=120); plt.show()

## 2. Fourier Ring Correlation: на яких масштабах два треки бачать те саме

Для двох карт $a$ і $b$ з Фур'є-образами $\hat a,\hat b$ і кільця частот $|k|\in$ кільце $r$:
$$\mathrm{FRC}(r)=\frac{\sum_{k\in r}\mathrm{Re}\,\hat a(k)\,\hat b^*(k)}{\sqrt{\sum_{k\in r}|\hat a(k)|^2\;\sum_{k\in r}|\hat b(k)|^2}}$$
Якщо $a=s+n_a$, $b=s+n_b$ зі спільним сигналом $s$ і незалежними шумами, то $\mathrm{FRC}(r)\approx \mathrm{SNR}(r)/(1+\mathrm{SNR}(r))$, тобто $\mathrm{SNR}(r)=\mathrm{FRC}/(1-\mathrm{FRC})$.

**Сезонна зміна** кожного треку: медіана вересня–жовтня мінус медіана червня–липня, лише основна версія обробки (1.0.11, див. v3). Статична різниця геометрії треків у такій зміні скорочується.

Пороги: **3σ** (van Heel, $3/\sqrt{N_r/2}$); **нуль-модель** (карту другого треку циклічно зсунуто на ≥ 12 км, p95 з 30 зсувів); **1/7** — критерій, за яким у кріо-ЕМ визначають роздільність.

In [ ]:
# Найбільший квадрат, де обидва треки валідні на ≥ 97 %
ii = np.pad(BOTH.astype('f8').cumsum(0).cumsum(1), ((1, 0), (1, 0))); best = None
for s_ in range(1200, 200, -20):
    for r0 in range(0, H - s_, 20):
        for c0 in range(0, W - s_, 20):
            if (ii[r0 + s_, c0 + s_] - ii[r0, c0 + s_] - ii[r0 + s_, c0] + ii[r0, c0]) / s_**2 >= .97: best = (r0, c0, s_); break
        if best: break
    if best: break
R0, C0, N = best; N = N // 2 * 2; WIN = (slice(R0, R0 + N), slice(C0, C0 + N))
print(f'Квадрат для FRC: {N} × {N} пікселів = {N * PX_KM:.1f} × {N * PX_KM:.1f} км')
HANN = np.outer(np.hanning(N), np.hanning(N))
f1 = np.fft.fftfreq(N, d=PX_KM); KX, KY = np.meshgrid(f1, f1, indexing='ij'); K = np.hypot(KX, KY)
EDGES = np.geomspace(1 / (N * PX_KM) * 1.5, 1 / (2 * PX_KM), 40); KC = np.sqrt(EDGES[1:] * EDGES[:-1])
RING = np.digitize(K, EDGES) - 1; RING[(RING < 0) | (RING >= len(KC))] = len(KC)

def spec(a):
    a = a[WIN].astype('f8'); m = np.isfinite(a); a = np.where(m, a - np.nanmean(a), 0)
    return np.fft.fft2(a * HANN)
def ringsum(v): return np.bincount(RING.ravel(), v.ravel(), len(KC) + 1)[:len(KC)]
def frc(a, b):
    Fa, Fb = spec(a), spec(b)
    return ringsum((Fa * np.conj(Fb)).real) / np.sqrt(ringsum(np.abs(Fa)**2) * ringsum(np.abs(Fb)**2))
CNT = np.bincount(RING.ravel(), minlength=len(KC) + 1)[:len(KC)]
SIG3 = 3 / np.sqrt(CNT / 2)

def season(t, key):
    T = S[t]
    e = [i for i, tt in enumerate(T['time']) if tt.month in (6, 7) and T['ver'][i] == '1.0.11']
    l = [i for i, tt in enumerate(T['time']) if tt.month in (9, 10) and T['ver'][i] == '1.0.11']
    return np.nanmedian(T[key][l], 0) - np.nanmedian(T[key][e], 0), e, l

FRC, NULL95 = {}, {}
SEAS = {}
for key in ('hh', 'hv'):
    (a, ea, la), (b, eb, lb) = season(A_, key), season(B_, key); SEAS[key] = (a, b)
    FRC[key] = frc(a, b)
    nulls = [frc(a, np.roll(np.roll(b, rng.integers(150, H - 150), 0), rng.integers(150, W - 150), 1)) for _ in range(30)]
    NULL95[key] = np.percentile(nulls, 95, axis=0)
print('Вікна сезону:', A_, [f"{S[A_]['time'][i]:%d.%m}" for i in ea], '→', [f"{S[A_]['time'][i]:%d.%m}" for i in la], '|',
      B_, [f"{S[B_]['time'][i]:%d.%m}" for i in eb], '→', [f"{S[B_]['time'][i]:%d.%m}" for i in lb])

def resolution(f, thr):
    """Найдрібніший масштаб (км), на якому FRC ще вища за поріг thr (скаляр або крива)."""
    thr = np.broadcast_to(thr, f.shape); ok = np.isfinite(f) & (f > thr)
    return 1 / KC[ok].max() if ok.any() else np.nan

BANDS = [('10–40 км', 1 / 40, 1 / 10), ('3–10 км', 1 / 10, 1 / 3), ('1–3 км', 1 / 3, 1), ('300 м–1 км', 1, 1 / .3), ('160–300 м', 1 / .3, 1 / .16)]
rows = []
for key in ('hh', 'hv'):
    r = {'поляр.': key.upper()}
    for nm, lo, hi in BANDS:
        sel = (KC >= lo) & (KC < hi); fv = np.nanmean(FRC[key][sel])
        r[f'FRC {nm}'] = fv; r[f'SNR {nm}'] = fv / (1 - fv) if fv < 1 else np.inf
    r['роздільність за 1/7, км'] = resolution(FRC[key], 1 / 7)
    r['роздільність за 3σ, км'] = resolution(FRC[key], SIG3)
    r['роздільність за нуль-моделлю, км'] = resolution(FRC[key], NULL95[key])
    rows.append(r)
FRC_TABLE = pd.DataFrame(rows); display(FRC_TABLE.round(3))

fig, ax = plt.subplots(figsize=(10, 5.2), constrained_layout=True)
for key, c in (('hh', 'tab:blue'), ('hv', 'tab:red')):
    ax.semilogx(KC, FRC[key], '-o', ms=3, c=c, label=f'сезонна зміна {key.upper()}: два треки')
    ax.semilogx(KC, NULL95[key], ':', c=c, label=f'нуль-модель p95 ({key.upper()})')
ax.semilogx(KC, SIG3, 'k--', lw=.8, label='3σ (van Heel)')
for key, c in (('hh', 'tab:blue'), ('hv', 'tab:red')):
    rr = resolution(FRC[key], 1 / 7)
    if np.isfinite(rr): ax.axvline(1 / rr, c=c, lw=.6, ls='--'); ax.text(1 / rr, 0.93, f' {key.upper()}: {rr * 1000:.0f} м', color=c, fontsize=8); ax.axhline(1 / 7, c='gray', ls='-.', lw=.8, label='1/7')
ax.axhline(0, c='k', lw=.4); ax.set(xlabel='просторова частота, цикли/км', ylabel='FRC', ylim=(-0.1, 1), title='Fourier Ring Correlation сезонних змін двох незалежних треків NISAR')
sec = ax.secondary_xaxis('top', functions=(lambda k: 1 / np.maximum(k, 1e-6), lambda s: 1 / np.maximum(s, 1e-6))); sec.set_xlabel('масштаб, км')
ax.legend(fontsize=8, loc='lower left'); fig.savefig(OUT / 'frc_season.png', dpi=140); plt.show()

## 3. Якість кожного тижня: FRC майже одночасних знімків

Тепер порівнюємо не зміни, а самі знімки HH з різницею в 3 доби. Висока FRC на масштабі полів означає, що того тижня обидва радари бачили поверхню. Низька — що її закрили перешкоди хоча б в одному треку. Статичні відмінності геометрії (висхідний проти низхідного) теж трохи знижують FRC, але однаково на всіх датах.

In [ ]:
rows = []; cm = plt.cm.viridis(np.linspace(0, 1, len(PAIRS)))
fig, axs = plt.subplots(1, 2, figsize=(16, 5), constrained_layout=True)
for (i, j), c in zip(PAIRS, cm):
    f = frc(S[A_]['hh'][i], S[B_]['hh'][j]); lab = f"{S[A_]['time'][i]:%d.%m}/{S[B_]['time'][j]:%d.%m}"
    axs[0].semilogx(KC, f, c=c, label=lab)
    field = (KC >= 1 / 3) & (KC < 1 / .3)
    rows.append({'пара': lab, 'дата': S[A_]['time'][i], 'FRC 3–10 км': np.nanmean(f[(KC >= .1) & (KC < 1 / 3)]), 'FRC поля 0.3–3 км': np.nanmean(f[field]),
                 'SNR поля': np.nanmean(f[field]) / (1 - np.nanmean(f[field]))})
WEEK = pd.DataFrame(rows)
axs[0].axhline(1 / 7, c='gray', ls='-.', lw=.8); axs[0].set(xlabel='цикли/км', ylabel='FRC', title='FRC знімків HH майже одночасних дат', ylim=(-0.1, 1)); axs[0].legend(fontsize=7, ncol=2)
axs[1].plot(WEEK['дата'], WEEK['FRC поля 0.3–3 км'], 'o-', label='масштаб полів 0.3–3 км'); axs[1].plot(WEEK['дата'], WEEK['FRC 3–10 км'], 's--', label='3–10 км')
axs[1].set(ylabel='FRC', title='Наскільки два радари бачили одне й те саме, по тижнях'); axs[1].legend()
fig.savefig(OUT / 'frc_by_week.png', dpi=130); plt.show()
display(WEEK.drop(columns='дата').round(3))

## 4. Оптимальне злиття двох треків і карта шуму

У кріо-ЕМ карту з двох половин фільтрують вагою $w(k)=\sqrt{\mathrm{FSC}_{full}}$, де $\mathrm{FSC}_{full}=2\,\mathrm{FRC}/(1+\mathrm{FRC})$ — оцінка кореляції середнього з істинним сигналом (Rosenthal & Henderson 2003). Масштаби, на яких треки незгодні, автоматично гасяться.
* **Сигнал:** $\mathcal F^{-1}\big[w(k)\,(\hat a+\hat b)/2\big]$ — сезонна зміна, у якій обидва треки згодні.
* **Шум:** $(a-b)/2$ — те, у чому вони розходяться (перешкоди, спекл, геометрія).

Масштаби, довші за вікно (> ~40 км), фільтр обнуляє: злита карта показує зміну **відносно регіонального фону**. Тому загальне «посиніння» сирого середнього на ній зникає.

In [ ]:
def wiener_fuse(a, b, f):
    w_ring = np.sqrt(np.clip(2 * f / (1 + f), 0, 1)); w = np.append(w_ring, 0)[RING]
    Fa, Fb = spec(a), spec(b)
    fused = np.real(np.fft.ifft2(w * (Fa + Fb) / 2)) / np.maximum(HANN, 1e-3)
    return fused, w
fused, wmap = wiener_fuse(*SEAS['hh'], FRC['hh'])
a, b = SEAS['hh']; noise = (a - b)[WIN] / 2
inner = (slice(N // 8, N - N // 8), slice(N // 8, N - N // 8))       # прибираємо краї, де вікно Ганна ≈ 0
ext = [X[C0 + N // 8] / 1e3, X[C0 + N - N // 8] / 1e3, Y[R0 + N - N // 8] / 1e3, Y[R0 + N // 8] / 1e3]
fig, axs = plt.subplots(1, 3, figsize=(19, 6), constrained_layout=True)
q = axs[0].imshow(((a + b) / 2)[WIN][inner], cmap='coolwarm', vmin=-5, vmax=5, extent=ext); axs[0].set_title('середнє двох треків (сире)')
axs[1].imshow(fused[inner], cmap='coolwarm', vmin=-5, vmax=5, extent=ext); axs[1].set_title('злиття з FRC-фільтром: спільний сигнал' + chr(10) + '(відносно регіонального фону, масштаби < 40 км)')
axs[2].imshow(noise[inner], cmap='coolwarm', vmin=-5, vmax=5, extent=ext); axs[2].set_title('шум: (143_A − 108_D) / 2')
fig.colorbar(q, ax=axs, shrink=.7, label='сезонна зміна HH (вер–жовт мінус черв–лип), дБ')
fig.savefig(OUT / 'fusion_signal_noise.png', dpi=130); plt.show()
sig_var, noise_var = np.nanvar(fused[inner]), np.nanvar(noise[inner])
print(f'Дисперсія: спільний сигнал {sig_var:.2f} дБ², шум (половина різниці) {noise_var:.2f} дБ², відношення {sig_var / noise_var:.1f}')

## 5. Де на ділянці радар бачить правду: локальна узгодженість

Аналог локальної роздільності в кріо-ЕМ (ResMap): ділимо ділянку на вікна 10 × 10 км з кроком 5 км і в кожному рахуємо кореляцію сезонних змін двох треків у смузі масштабів полів (0.3–3 км). Висока — радар бачить зміни полів; низька — там переважають перешкоди або нічого не змінювалось.

In [ ]:
TS_ = 126; STEP = 63
hw = np.outer(np.hanning(TS_), np.hanning(TS_)); g = np.fft.fftfreq(TS_, d=PX_KM); kk = np.hypot(*np.meshgrid(g, g, indexing='ij'))
band = (kk >= 1 / 3) & (kk <= 1 / .3)
def band_corr(x, y):
    m = np.isfinite(x) & np.isfinite(y)
    if m.mean() < .8: return np.nan
    x = np.where(m, x - np.nanmean(x), 0) * hw; y = np.where(m, y - np.nanmean(y), 0) * hw
    Fx, Fy = np.fft.fft2(x)[band], np.fft.fft2(y)[band]
    return float((Fx * np.conj(Fy)).real.sum() / np.sqrt((np.abs(Fx)**2).sum() * (np.abs(Fy)**2).sum()))
a, b = SEAS['hh']
rs = list(range(0, H - TS_, STEP)); cs = list(range(0, W - TS_, STEP))
LOC = np.full((len(rs), len(cs)), np.nan)
for ii_, r0 in enumerate(rs):
    for jj, c0 in enumerate(cs):
        LOC[ii_, jj] = band_corr(a[r0:r0 + TS_, c0:c0 + TS_], b[r0:r0 + TS_, c0:c0 + TS_])
loc_ext = [X[0] / 1e3 + TS_ / 2 * PX_KM - STEP / 2 * PX_KM, X[0] / 1e3 + (cs[-1] + TS_ / 2 + STEP / 2) * PX_KM,
           Y[0] / 1e3 - (rs[-1] + TS_ / 2 + STEP / 2) * PX_KM, Y[0] / 1e3 - TS_ / 2 * PX_KM + STEP / 2 * PX_KM]
fig, axs = plt.subplots(1, 2, figsize=(17, 6), constrained_layout=True)
axs[0].imshow(np.nanmedian(S[A_]['hh'], 0), cmap='gray', vmin=-18, vmax=-2, extent=EXTENT)
q = axs[0].imshow(LOC, cmap='magma', vmin=0, vmax=0.8, extent=loc_ext, alpha=.75, interpolation='bilinear')
fig.colorbar(q, ax=axs[0], shrink=.8, label='узгодженість змін двох треків (масштаб полів)'); axs[0].set_title('Де NISAR бачить справжні зміни полів')
axs[1].hist(LOC[np.isfinite(LOC)], bins=30, color='tab:purple'); axs[1].set(xlabel='узгодженість у вікні 10 км', ylabel='вікон', title=f'Розподіл: медіана {np.nanmedian(LOC):.2f}, {np.mean(LOC[np.isfinite(LOC)] > .5):.0%} вікон > 0.5')
fig.savefig(OUT / 'local_agreement.png', dpi=130); plt.show()

## 6. Ритм: що змінюється однаково в обох треках протягом сезону

Одиниця аналізу — **блок 400 × 400 м** (5 × 5 пікселів, усереднення в лінійній потужності): спекл гаситься, а поля ще помітні. Для кожного блоку беремо **8 пар майже одночасних дат** (лише основна версія обробки: спільний стрибок версії наприкінці сезону дав би штучний спільний ритм) і рахуємо кореляцію Пірсона рядів HH двох треків $r$.
* **Нуль:** 5000 випадкових перестановок порядку дат другого треку; p = частка перестановок з r не меншим. Мінімальне можливе p = 1/5001.
* **Частка справжнього ритму за Storey:** $\hat\pi_1 = 1-\hat\pi_0$, $\hat\pi_0=\#\{p>\lambda\}/((1-\lambda)m)$, λ = 0.5. Оцінка не залежить від вибору порога.
* **FDR 5 %** (Benjamini–Hochberg): карта блоків, де ритм підтверджено. З 8 датами тест має обмежену силу, тож додатково оцінюємо **частку хибних серед блоків з p < 0.05**: $\widehat{\mathrm{FDR}}=\hat\pi_0\cdot 0.05/\Pr(p<0.05)$ (Storey 2002).

Далі — **SVD спільного сигналу** підтверджених блоків: середнє двох треків після z-нормування кожного ряду. Праві сингулярні вектори — головні **часові моди** (типові сезонні криві), ліві — вага моди в кожному блоці.

In [ ]:
RP = [(i, j) for i, j in PAIRS if S[A_]['ver'][i] == '1.0.11' and S[B_]['ver'][j] == '1.0.11']
print('Пари для ритму:', ', '.join(f"{S[A_]['time'][i]:%d.%m}/{S[B_]['time'][j]:%d.%m}" for i, j in RP))
BK = 5; HB, WB = H // BK, W // BK
def blk_mean_db(a):
    lin = 10 ** (a[:HB * BK, :WB * BK] / 10)
    return 10 * np.log10(np.nanmean(lin.reshape(HB, BK, WB, BK), axis=(1, 3)))
def blk_frac(m): return m[:HB * BK, :WB * BK].reshape(HB, BK, WB, BK).mean(axis=(1, 3))
TA = np.stack([blk_mean_db(S[A_]['hh'][i]) for i, _ in RP]).astype('f8')
TB = np.stack([blk_mean_db(S[B_]['hh'][j]) for _, j in RP]).astype('f8')
VB = (blk_frac(BOTH) == 1) & np.isfinite(TA).all(0) & np.isfinite(TB).all(0)
CLASSES = {'рілля': CROP, 'дно: дерева': BED & (WC == 10), 'дно: трави/чагарники/болото': BED & np.isin(WC, (20, 30, 90)),
           'забудова': (WC == 50) & LAND, 'трави поза дном': (WC == 30) & ~BED}
CLB = {nm: (blk_frac(m) >= 0.8) & VB for nm, m in CLASSES.items()}     # блок належить класу, якщо ≥ 80 % пікселів
def zs(Z): Z = Z - Z.mean(0); return Z / np.maximum(np.sqrt((Z**2).sum(0)), 1e-9)
ZA, ZB = zs(np.where(VB, TA, 0)), zs(np.where(VB, TB, 0))
R = (ZA * ZB).sum(0)
NPERM = 5000; exceed = np.zeros((HB, WB), 'i4')
for _ in range(NPERM):
    exceed += ((ZA * ZB[rng.permutation(len(RP))]).sum(0) >= R)
P = (exceed + 1) / (NPERM + 1)
def storey_pi1(p, lam=0.5): return max(0.0, 1 - np.mean(p > lam) / (1 - lam))
pv = np.sort(P[VB]); m_ = pv.size; ok = pv <= 0.05 * np.arange(1, m_ + 1) / m_
P_THR = pv[ok.nonzero()[0].max()] if ok.any() else 0.0
RHYTHM = VB & (P <= P_THR)
BKM2 = (BK * PX_KM) ** 2
rows = [{'клас': 'усі блоки', 'блоків': int(VB.sum()), 'км²': VB.sum() * BKM2, 'медіана r': np.median(R[VB]), 'частка p<0.05': np.mean(P[VB] < .05),
         'π₁ Storey (частка з ритмом)': storey_pi1(P[VB]), 'ритм за FDR, км²': RHYTHM.sum() * BKM2}]
for nm, m in CLB.items():
    if m.sum() < 30: continue
    rows.append({'клас': nm, 'блоків': int(m.sum()), 'км²': m.sum() * BKM2, 'медіана r': np.median(R[m]), 'частка p<0.05': np.mean(P[m] < .05),
                 'π₁ Storey (частка з ритмом)': storey_pi1(P[m]), 'ритм за FDR, км²': (RHYTHM & m).sum() * BKM2})
RHYTHM_TABLE = pd.DataFrame(rows)
RHYTHM_TABLE['FDR серед p<0.05 (Storey)'] = (1 - RHYTHM_TABLE['π₁ Storey (частка з ритмом)']) * 0.05 / RHYTHM_TABLE['частка p<0.05']
display(RHYTHM_TABLE.round(3))
print(f'Поріг FDR 5 %: p ≤ {P_THR:.4f}')

ext_b = [EXTENT[0], EXTENT[0] + WB * BK * PX_KM, EXTENT[3] - HB * BK * PX_KM, EXTENT[3]]
fig, axs = plt.subplots(1, 2, figsize=(17, 6), constrained_layout=True)
q = axs[0].imshow(np.where(VB, R, np.nan), cmap='RdBu_r', vmin=-1, vmax=1, extent=ext_b, interpolation='nearest')
fig.colorbar(q, ax=axs[0], shrink=.8, label=f'r між треками ({len(RP)} пар дат), блоки 400 м'); axs[0].set_title('Кореляція сезонних рядів HH двох треків')
axs[1].imshow(np.nanmedian(S[A_]['hh'], 0), cmap='gray', vmin=-18, vmax=-2, extent=EXTENT)
axs[1].imshow(np.ma.masked_where(~RHYTHM, RHYTHM), cmap=ListedColormap(['#ffb000']), extent=ext_b, interpolation='nearest')
axs[1].set_title(f'Ритм підтверджено: перестановки + FDR 5 % ({RHYTHM.sum() * BKM2:,.0f} км²)')
fig.savefig(OUT / 'rhythm_map.png', dpi=130); plt.show()

# SVD спільного ритму
selb = RHYTHM if RHYTHM.sum() >= 200 else VB & (P < .05)
print(f'SVD на {selb.sum()} блоках ({"FDR" if selb is RHYTHM else "p < 0.05, бо за FDR замало"})')
M = ((ZA[:, selb] + ZB[:, selb]) / 2).T
U, s, Vt = np.linalg.svd(M - M.mean(0), full_matrices=False)
ev = s**2 / (s**2).sum()
for k in range(3):
    if Vt[k][np.argmax(np.abs(Vt[k]))] < 0: Vt[k] *= -1; U[:, k] *= -1
dates = [S[A_]['time'][i] + (S[B_]['time'][j] - S[A_]['time'][i]) / 2 for i, j in RP]
Lb = np.full((HB, WB, 3), np.nan); Lb[selb] = U[:, :3] * s[:3]
fig, axs = plt.subplots(1, 2, figsize=(17, 5.5), constrained_layout=True)
for k, c in zip(range(3), ('tab:blue', 'tab:orange', 'tab:green')):
    axs[0].plot(dates, Vt[k], 'o-', c=c, label=f'мода {k + 1}: {ev[k]:.0%} дисперсії')
yl = axs[0].get_ylim()
for lo, hi, nm in (('2026-06-25', '2026-07-31', 'пшениця/ячмінь'), ('2026-08-25', '2026-09-30', 'соняшник'), ('2026-09-15', '2026-10-31', 'кукурудза')):
    axs[0].axvspan(pd.Timestamp(lo), pd.Timestamp(hi), alpha=.07, color='k'); axs[0].text(pd.Timestamp(lo), yl[1] * .92, nm, fontsize=8)
axs[0].axhline(0, c='k', lw=.5); axs[0].set(title='Часові моди спільного ритму (SVD)', ylabel='вага дати'); axs[0].legend(fontsize=8)
rgb = np.clip(0.5 + Lb / (2.5 * np.nanstd(Lb, axis=(0, 1))), 0, 1); rgb[~selb] = 0.12
axs[1].imshow(rgb, extent=ext_b, interpolation='nearest'); axs[1].set_title('Вага мод: R = мода 1, G = мода 2, B = мода 3 (темне — ритму не підтверджено)')
fig.savefig(OUT / 'rhythm_modes.png', dpi=130); plt.show()
print('Частка дисперсії перших трьох мод:', np.round(ev[:3], 3))

# Яка мода — перешкода? Незалежний вимірник: рівень HH над постійною водою на кожну дату (як у v3)
D_ = {t: np.stack([np.load(CACHE / f"dswx_s1_{tt:%Y%m%dT%H%M}.npz")['mos'] for tt in S[t]['time']]) for t in TRACKS}
wat = sum(np.isin(D_[t], (1, 3)).sum(0) for t in TRACKS); val = sum((D_[t] < 250).sum(0) for t in TRACKS)
CORE = binary_erosion((val >= 10) & (wat / np.maximum(val, 1) >= .9), iterations=3)
floor = np.array([(np.nanmedian(S[A_]['hh'][i][CORE]) + np.nanmedian(S[B_]['hh'][j][CORE])) / 2 for i, j in RP])
# Зона плями перешкод (як у v3): багатодатний HV-фон суші вищий за медіану AOI більш ніж на 3 дБ
def blk_med(a, B_=25):
    h, w = (H // B_) * B_, (W // B_) * B_
    v = a[:h, :w].reshape(h // B_, B_, w // B_, B_).transpose(0, 2, 1, 3).reshape(h // B_, w // B_, -1)
    return np.nanmedian(v, -1)
RFI_ZONE = np.zeros((H, W), bool)
for t in TRACKS:
    bg = blk_med(np.nanmedian(np.where(LAND, S[t]['hv'], np.nan), 0)); up = np.kron(bg, np.ones((25, 25)))
    z = np.zeros((H, W)); z[:up.shape[0], :up.shape[1]] = up; z[up.shape[0]:] = z[up.shape[0] - 1]; z[:, up.shape[1]:] = z[:, up.shape[1] - 1:up.shape[1]]
    RFI_ZONE |= (z > np.nanmedian(bg) + 3) & (np.isfinite(S[t]['hh']).mean(0) > .5)
RZB = blk_frac(RFI_ZONE) > .5
MODES = []
for k in range(3):
    r_floor = np.corrcoef(Vt[k], floor)[0, 1]
    w2 = Lb[..., k] ** 2; inz = np.nansum(w2[RZB & selb]) / np.nansum(w2[selb])
    MODES.append({'мода': k + 1, 'частка дисперсії': ev[k], 'r з рівнем перешкод на воді': r_floor,
                  'частка енергії моди в плямі перешкод': inz, 'площа плями серед ритмічних блоків': (RZB & selb).sum() / selb.sum(),
                  'тлумачення': 'перешкода (часовий хід = рівень перешкод)' if abs(r_floor) >= .6 else 'поверхня / змішане'})
MODES = pd.DataFrame(MODES); display(MODES.round(3))
fig, ax = plt.subplots(figsize=(10, 3.6), constrained_layout=True)
k_rfi = int(MODES.loc[MODES['r з рівнем перешкод на воді'].abs().idxmax(), 'мода']) - 1
ax.plot(dates, (Vt[k_rfi] - Vt[k_rfi].mean()) / Vt[k_rfi].std(), 'o-', label=f'мода {k_rfi + 1} SVD (нормовано)')
ax.plot(dates, np.sign(MODES.loc[k_rfi, 'r з рівнем перешкод на воді']) * (floor - floor.mean()) / floor.std(), 's--', c='k', label='рівень HH над постійною водою, обернений знак (незалежний вимір перешкод; вгору = перешкод менше)')
ax.legend(); ax.set_title(f"SVD сама знайшла перешкоду: r = {MODES.loc[k_rfi, 'r з рівнем перешкод на воді']:.2f}")
fig.savefig(OUT / 'svd_finds_rfi.png', dpi=130); plt.show()

# Ритм поверхні: лише блоки поза плямою перешкод
rows = []
for nm, m in {'усі блоки': VB, **CLB}.items():
    for zone, zm in (('у плямі', RZB), ('поза плямою', ~RZB)):
        mm = m & zm
        if mm.sum() < 30: continue
        pi1 = storey_pi1(P[mm]); f05 = np.mean(P[mm] < .05)
        rows.append({'клас': nm, 'зона': zone, 'блоків': int(mm.sum()), 'медіана r': np.median(R[mm]), 'π₁ Storey': pi1, 'FDR серед p<0.05': (1 - pi1) * .05 / max(f05, 1e-9)})
RHYTHM_ZONE = pd.DataFrame(rows); display(RHYTHM_ZONE.round(3))

rows = []
for nm, m in CLB.items():
    mm = m & selb
    if mm.sum() < 20: continue
    rows.append({'клас': nm, 'блоків з ритмом': int(mm.sum()), **{f'мода {k + 1}': np.nanmedian(Lb[..., k][mm]) for k in range(3)}})
MODE_BY_CLASS = pd.DataFrame(rows); display(MODE_BY_CLASS.round(3))
if len(MODE_BY_CLASS):
    fig, ax = plt.subplots(figsize=(10, 3.8), constrained_layout=True); xx = np.arange(len(MODE_BY_CLASS))
    for k, c in zip(range(3), ('tab:blue', 'tab:orange', 'tab:green')):
        ax.bar(xx + (k - 1) * .27, MODE_BY_CLASS[f'мода {k + 1}'], .27, color=c, label=f'мода {k + 1}')
    ax.set_xticks(xx); ax.set_xticklabels(MODE_BY_CLASS['клас'], fontsize=8); ax.axhline(0, c='k', lw=.5); ax.legend()
    ax.set_title('Ритмічний «підпис» класів: медіанна вага кожної моди'); fig.savefig(OUT / 'rhythm_signature_by_class.png', dpi=130); plt.show()

## 7. Що не спрацювало: віднімати перешкоду моделлю

Ми перевірили модель $Y_{HH}=S+N$, $Y_{HV}=qS+\kappa N$ з гладкою в просторі $N$, яку оцінювали ітеративно (κ — з постійної води, q — з медіани відношення HV/HH поверхні). Вода після цього стала темною, а HH − HV ріллі — стабільним (~5.5 дБ), але **це наслідок самої моделі**. Незалежна перевірка показала протилежне: кореляція сезонних змін двох треків **впала** (HH рілля 0.69 → 0.48, HV 0.50 → 0.33), а omnibus-тест на забудові позначав зміну у 96–99 % пікселів і до, і після. Тому в цьому ноутбуці перешкоду не моделюємо, а **дозволяємо двом трекам її виявити**. Код прототипу: `scratchpad/proto_sep.py` (не входить у ноутбук).

## 8. Підсумок

In [ ]:
fr = FRC_TABLE.set_index('поляр.')
L_ = [
    f"1. Сезонна зміна HH двох незалежних треків збігається: FRC {fr.loc['HH', 'FRC 3–10 км']:.2f} на 3–10 км, {fr.loc['HH', 'FRC 1–3 км']:.2f} на 1–3 км, "
    f"{fr.loc['HH', 'FRC 300 м–1 км']:.2f} на 300 м–1 км (SNR {fr.loc['HH', 'SNR 300 м–1 км']:.1f}). Роздільність за 1/7: {fr.loc['HH', 'роздільність за 1/7, км']:.2f} км; "
    f"за 3σ: {fr.loc['HH', 'роздільність за 3σ, км']:.2f} км; за нуль-моделлю: {fr.loc['HH', 'роздільність за нуль-моделлю, км']:.2f} км.",
    f"2. HV слабший: FRC {fr.loc['HV', 'FRC 1–3 км']:.2f} на 1–3 км, {fr.loc['HV', 'FRC 300 м–1 км']:.2f} на 300 м–1 км; роздільність за 1/7 {fr.loc['HV', 'роздільність за 1/7, км']:.2f} км.",
    f"3. По тижнях FRC на масштабі полів від {WEEK['FRC поля 0.3–3 км'].min():.2f} ({WEEK.loc[WEEK['FRC поля 0.3–3 км'].idxmin(), 'пара']}) "
    f"до {WEEK['FRC поля 0.3–3 км'].max():.2f} ({WEEK.loc[WEEK['FRC поля 0.3–3 км'].idxmax(), 'пара']}).",
    f"4. Злиття з FRC-фільтром: дисперсія спільного сигналу {sig_var:.2f} дБ² проти шуму {noise_var:.2f} дБ² (×{sig_var / noise_var:.1f}).",
    f"5. Локальна узгодженість на масштабі полів: медіана {np.nanmedian(LOC):.2f}, > 0.5 у {np.mean(LOC[np.isfinite(LOC)] > .5):.0%} вікон 10 км.",
    "6. Ритм (блоки 400 м, 8 пар дат, перестановки): " + '; '.join(f"{r['клас']}: медіана r {r['медіана r']:.2f}, частка з ритмом π₁ {r['π₁ Storey (частка з ритмом)']:.0%}, FDR серед p<0.05 ≈ {r['FDR серед p<0.05 (Storey)']:.0%}" for _, r in RHYTHM_TABLE.iterrows()) + '.',
    f"7. Три часові моди спільного ритму пояснюють {ev[:3].sum():.0%} дисперсії ({', '.join(f'{v:.0%}' for v in ev[:3])}). "
    + '; '.join(f"мода {int(r['мода'])}: r з рівнем перешкод {r['r з рівнем перешкод на воді']:+.2f}, у плямі {r['частка енергії моди в плямі перешкод']:.0%} енергії → {r['тлумачення']}" for _, r in MODES.iterrows()) + '.',
    "8. Ритм поза плямою перешкод: " + '; '.join(f"{r['клас']}: π₁ {r['π₁ Storey']:.0%} (медіана r {r['медіана r']:.2f})" for _, r in RHYTHM_ZONE[RHYTHM_ZONE['зона'] == 'поза плямою'].iterrows()) + '.',
]
print('\n'.join(L_)); (OUT / 'summary.txt').write_text('\n'.join(L_), encoding='utf-8')
FACTS = {'frc_table': FRC_TABLE.to_dict('records'), 'frc_curve': {'k_cyc_per_km': KC.tolist(), 'hh': FRC['hh'].tolist(), 'hv': FRC['hv'].tolist(),
         'null95_hh': NULL95['hh'].tolist(), 'sigma3': SIG3.tolist()}, 'weekly': WEEK.assign(дата=WEEK['дата'].astype(str)).to_dict('records'),
         'rhythm_table': RHYTHM_TABLE.to_dict('records'), 'svd_explained': ev[:5].tolist(), 'svd_modes_vs_rfi': MODES.to_dict('records'), 'rhythm_by_zone': RHYTHM_ZONE.to_dict('records'), 'svd_modes': Vt[:3].tolist(), 'mode_dates': [str(d.date()) for d in dates]}
(OUT / 'facts.json').write_text(json.dumps(FACTS, ensure_ascii=False, indent=1, default=str), encoding='utf-8')
print('Збережено:', OUT)

## 9. Висновки: що два спостерігачі можуть і чого не можуть

**Простір: працює.** Сезонна зміна HH (вересень–жовтень проти червня–липня) у двох незалежних треках збігається **за всіма масштабами від десятків кілометрів до ~200 м** (FRC > 1/7 до 200 м, > 3σ до 170 м). На масштабі полів 300 м–1 км FRC = 0.53, тобто SNR ≈ 1.1; на 1–3 км SNR ≈ 2. HV відстає: роздільність ~420 м. Ефективна роздільність змін ~200 м — це розмір поля, а не піксель 80 м. Перешкоди її не знищили, бо вони гладкі на десятках кілометрів, а поля — ні.

**Тижні: видно, коли радар «прозрів».** FRC майже одночасних знімків на масштабі полів зростає з 0.45 (кінець червня) до 0.73 (кінець вересня). Найкраще — 23.08/20.08 і 28.09/25.09: ті самі тижні, коли перешкоди на воді найслабші (v3).

**Місце: видно, де радар бачить правду.** Узгодженість змін у вікнах 10 км > 0.5 у двох третинах ділянки. Провали збігаються з плямою перешкод на північному сході та з окремими локальними плямами.

**Час: не працює так просто, і це наша найцікавіша знахідка.** Кореляція часових рядів двох треків здебільшого сидить **у плямі перешкод**: π₁ = 73 % блоків там проти 26 % поза нею (рілля поза плямою — 14 %). Перша мода SVD (41 % дисперсії) повторює незалежно виміряний рівень перешкод на воді з r = −0.80. Тобто **обидва треки бачать те саме наземне джерело перешкод**, і в часі «двоє спостерігачів» дивляться на одну й ту саму лампу. SVD відділяє цю лампу сліпо, без жодної моделі перешкоди: це другий незалежний спосіб її виявити.

**Що з цього брати в продукт**
* Головна карта — **злиття двох треків із FRC-фільтром** (розділ 4) разом із **картою локальної узгодженості** (розділ 5) як шаром довіри.
* Головна крива — **FRC за масштабом** (розділ 2): одна картинка, яка відповідає на питання «що NISAR бачить над Каховкою і до якої дрібності».
* Часові ряди окремих пікселів у плямі не тлумачимо; моду 1 SVD показуємо як «відбиток перешкоди».

**Звідки методи:** van Heel (1987), Saxton & Baumeister (1982) — FRC/FSC; Rosenthal & Henderson (2003) — фільтр $\sqrt{2\mathrm{FRC}/(1+\mathrm{FRC})}$ і критерій 1/7; Storey (2002) — $\hat\pi_0$; Benjamini & Hochberg (1995) — FDR; ідея порівнювати незалежні половини даних замість моделювання шуму — з кріо-ЕМ, де її називають «gold standard».